# AF2·09 — End-to-End Toy AlphaFold

**The payoff of the whole series.** Eight rungs built the parts; this one wires them into a
single model that goes all the way from a **multiple sequence alignment to 3D coordinates**,
trained end-to-end, and folds proteins it has never seen — from their MSA alone.

Here is the entire pipeline you have built, now connected:

```
MSA  ──▶  Evoformer trunk  ──▶  pair representation  ──▶  distogram
 (01)      (02 axial attn        (03 triangle ops)         (04 head)
            + 03 + 04)                │
                                      ▼
                          structure module  ──▶  3D backbone
                          (05 frames, 06 IPA,
                           07 frame updates + FAPE)
```

Track A (rungs 01–04) turns the MSA into a geometrically-consistent pair representation and
a predicted distance map. Track B (rungs 05–08) turns that into a set of residue frames — a
structure — via invariant point attention and frame updates, scored by FAPE. We connect
them through the **distogram**: the trunk predicts distances, the structure module folds
them, and gradients flow through the whole thing so the two halves learn together.

This is a real AlphaFold in miniature. It will not be sub-ångström — that needs the full
architecture, deep MSAs, and enormous training — but every mechanism is the genuine article,
and the end-to-end result is unambiguous: **fed only an MSA, it folds a recognisable
backbone, far better than chance.**

**How to use this notebook:** implement the integration reps, make the checkpoints pass.
Solutions at the bottom. Trains in ~2 minutes on a laptop CPU (the whole pipeline, end to
end).

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0); rng = np.random.default_rng(0)
plt.rcParams['axes.spines.top'] = False; plt.rcParams['axes.spines.right'] = False
BLUE, GREEN, INK = '#2a78d6', '#008300', '#52514e'
Q = 20; L = 16; Nseq = 128; c = 48; cz = 24; NITER = 8

# ---- toy protein: 3D structure -> contacts -> coevolving MSA (+ phylogeny). all given. ----
def make_structure():
    pos = [np.zeros(3)]; d = rng.normal(0, 1, 3); d /= np.linalg.norm(d)
    for _ in range(L - 1):
        d = d + rng.normal(0, 0.5, 3); d /= np.linalg.norm(d); pos.append(pos[-1] + d - 0.03 * pos[-1])
    X = np.array(pos); X -= X.mean(0); return X.astype(np.float32)
DIST_BINS = torch.linspace(1.5, 9.0, 15)

def make_example():
    '''-> (MSA [Nseq,L], true coords [L,3], true distogram bins [L,L], APC coevolution [L,L]).'''
    X = make_structure(); Xt = torch.tensor(X)
    D = np.sqrt(((X[:, None] - X[None]) ** 2).sum(-1))
    Cm = (D < 2.5) & (np.abs(np.arange(L)[:, None] - np.arange(L)[None]) >= 3)
    con = set(map(tuple, np.argwhere(np.triu(Cm))))
    prof = np.array([rng.dirichlet(np.ones(Q) * 0.8) for _ in range(L)]); perm = {p: rng.permutation(Q) for p in con}
    m = np.zeros((Nseq, L), int)
    for p in range(L): m[:, p] = rng.choice(Q, size=Nseq, p=prof[p])
    block = rng.choice(L, size=6, replace=False); cons = [{cc: int(rng.integers(Q)) for cc in block} for _ in range(4)]; cl = rng.integers(4, size=Nseq)
    for cc in block:
        for g in range(4):
            hh = (cl == g) & (rng.random(Nseq) < 0.9); m[hh, cc] = cons[g][cc]
    for (i, j) in sorted(con):
        cp = rng.random(Nseq) < 0.8; m[cp, j] = perm[(i, j)][m[cp, i]]
    oh = np.eye(Q)[m]; ps = 0.5; fi = (oh.sum(0) + ps) / (Nseq + Q * ps); MI = np.zeros((L, L))
    for i in range(L):
        for j in range(i + 1, L):
            fij = (oh[:, i].T @ oh[:, j] + ps / Q) / (Nseq + ps); fij /= fij.sum()
            MI[i, j] = MI[j, i] = (fij * np.log(fij / (np.outer(fi[i], fi[j]) + 1e-12) + 1e-12)).sum()
    mm = MI.copy(); np.fill_diagonal(mm, 0); col = mm.sum(1, keepdims=True) / (L - 1)
    a = mm[~np.eye(L, dtype=bool)].mean(); APC = mm - (col @ col.T) / a; np.fill_diagonal(APC, 0)
    dbin = torch.bucketize(torch.cdist(Xt, Xt), DIST_BINS).clamp(max=15)
    return torch.tensor(m), Xt, dbin, torch.tensor(APC, dtype=torch.float32)

def true_frames(X):
    R = torch.zeros(L, 3, 3)
    for i in range(L):
        a = X[min(i+1, L-1)] - X[i]; b = X[max(i-1, 0)] - X[i]
        e1 = a/(a.norm()+1e-8); e2 = b-(e1@b)*e1; e2 = e2/(e2.norm()+1e-8)
        R[i] = torch.stack([e1, e2, torch.cross(e1, e2, dim=-1)], -1)
    return R

train = [make_example() for _ in range(160)]; test = [make_example() for _ in range(30)]
Rtrain = [true_frames(x[1]) for x in train]; Rtest = [true_frames(x[1]) for x in test]
print('%d train + %d test toy proteins | each is an MSA of %d sequences x %d residues'
      % (len(train), len(test), Nseq, L))

In [ ]:
# ---------- all machinery below is GIVEN — you built every piece in rungs 02-07 ----------
def hat(w):
    O = torch.zeros(*w.shape[:-1], 3, 3)
    O[...,0,1]=-w[...,2];O[...,0,2]=w[...,1];O[...,1,0]=w[...,2];O[...,1,2]=-w[...,0];O[...,2,0]=-w[...,1];O[...,2,1]=w[...,0]
    return O
def so3_exp(w):
    th=w.norm(dim=-1,keepdim=True).clamp_min(1e-8);K=hat(w/th);th=th[...,None];return torch.eye(3).expand_as(K)+torch.sin(th)*K+(1-torch.cos(th))*(K@K)
def fape(tp,Rp,tt,Rt,clamp=10.):
    lp=torch.einsum('lji,lkj->lki',Rp,tp[None]-tp[:,None]);lt=torch.einsum('lji,lkj->lki',Rt,tt[None]-tt[:,None]);return (lp-lt).norm(dim=-1).clamp(max=clamp).mean()
def rowattn(q,k,v,pb):d=q.size(-1);return F.softmax(q@k.transpose(-2,-1)/math.sqrt(d)+pb[None],-1)@v
def colattn(q,k,v):d=q.size(-1);return F.softmax(q@k.transpose(-2,-1)/math.sqrt(d),-1)@v
def triup(a,b):return torch.einsum('ikc,jkc->ijc',a,b)+torch.einsum('kic,kjc->ijc',a,b)
class Row(nn.Module):
    def __init__(s):super().__init__();s.q=nn.Linear(c,c,bias=False);s.k=nn.Linear(c,c,bias=False);s.v=nn.Linear(c,c,bias=False);s.b=nn.Linear(cz,4,bias=False);s.g=nn.Linear(c,c);s.o=nn.Linear(c,c);s.ln=nn.LayerNorm(c)
    def forward(s,m,z):
        N=m.shape[0];x=s.ln(m);sp=lambda t:t.view(N,L,4,c//4).permute(0,2,1,3)
        return m+s.o(rowattn(sp(s.q(x)),sp(s.k(x)),sp(s.v(x)),s.b(z).permute(2,0,1)).permute(0,2,1,3).reshape(N,L,c)*torch.sigmoid(s.g(x)))
class Col(nn.Module):
    def __init__(s):super().__init__();s.q=nn.Linear(c,c,bias=False);s.k=nn.Linear(c,c,bias=False);s.v=nn.Linear(c,c,bias=False);s.g=nn.Linear(c,c);s.o=nn.Linear(c,c);s.ln=nn.LayerNorm(c)
    def forward(s,m):
        mt=s.ln(m).transpose(0,1);N=mt.shape[1];sp=lambda t:t.view(L,N,4,c//4).permute(0,2,1,3)
        return m+(s.o(colattn(sp(s.q(mt)),sp(s.k(mt)),sp(s.v(mt))).permute(0,2,1,3).reshape(L,N,c)*torch.sigmoid(s.g(mt)))).transpose(0,1)
class OPM(nn.Module):
    def __init__(s):super().__init__();s.ln=nn.LayerNorm(c);s.p=nn.Linear(c*c,cz)
    def forward(s,m):x=s.ln(m);return s.p((torch.einsum('nic,njd->ijcd',x,x)/x.shape[0]).reshape(L,L,-1))
class Tri(nn.Module):
    def __init__(s):super().__init__();s.ln=nn.LayerNorm(cz);s.lno=nn.LayerNorm(cz);s.a=nn.Linear(cz,cz);s.b=nn.Linear(cz,cz);s.ag=nn.Linear(cz,cz);s.bg=nn.Linear(cz,cz);s.o=nn.Linear(cz,cz);s.og=nn.Linear(cz,cz)
    def forward(s,z):zz=s.ln(z);a=torch.sigmoid(s.ag(zz))*s.a(zz);b=torch.sigmoid(s.bg(zz))*s.b(zz);return z+torch.sigmoid(s.og(zz))*s.o(s.lno(triup(a,b)))
h,dd,Np,Npv=4,8,4,6
class IPA(nn.Module):
    def __init__(s):
        super().__init__();s.qs=nn.Linear(c,h*dd);s.ks=nn.Linear(c,h*dd);s.vs=nn.Linear(c,h*dd);s.qp=nn.Linear(c,h*Np*3);s.kp=nn.Linear(c,h*Np*3);s.vp=nn.Linear(c,h*Npv*3);s.bz=nn.Linear(cz,h);s.gamma=nn.Parameter(torch.zeros(h));s.out=nn.Linear(h*dd+h*cz+h*Npv*3+h*Npv,c)
    def forward(s,x,z,R,t):
        qs=s.qs(x).view(L,h,dd);ks=s.ks(x).view(L,h,dd);vs=s.vs(x).view(L,h,dd)
        g=lambda lp:torch.einsum('lij,lhpj->lhpi',R,lp)+t[:,None,None,:]
        Qg=g(s.qp(x).view(L,h,Np,3));Kg=g(s.kp(x).view(L,h,Np,3));Vg=g(s.vp(x).view(L,h,Npv,3))
        scal=torch.einsum('ihd,jhd->ijh',qs,ks)/math.sqrt(dd);d2=((Qg[:,None]-Kg[None])**2).sum(-1).sum(-1)
        a=F.softmax(scal+s.bz(z)-0.5*F.softplus(s.gamma)[None,None]*d2,1)
        o_s=torch.einsum('ijh,jhd->ihd',a,vs).reshape(L,h*dd);o_z=torch.einsum('ijh,ijz->ihz',a,z).reshape(L,h*cz)
        og=torch.einsum('ijh,jhpx->ihpx',a,Vg);ol=torch.einsum('lji,lhpj->lhpi',R,og-t[:,None,None,:])
        return s.out(torch.cat([o_s,o_z,ol.reshape(L,h*Npv*3),ol.norm(dim=-1).reshape(L,h*Npv)],-1))
print('trunk + structure-module layers ready (rungs 02-07).')

## Part 1 — the handoff: MSA → a single per-residue representation

The trunk carries an MSA representation `m` of shape `[N_seq, L, c]` — a feature vector per
(sequence, residue). But the structure module reasons about **one** thing per residue, not
one per sequence. So we pool the MSA down to a single per-residue representation to seed the
folding.

AlphaFold takes the row of the target sequence; we simply **average over sequences**, which
for our symmetric toy MSA is the natural summary. This pooled vector is what becomes each
residue's starting single representation in the structure module.

### Rep 1 — `pool_single(m)`
Average the MSA representation `m` `[N_seq, L, c]` over the sequence axis to get the single
representation `[L, c]`.

In [ ]:
def pool_single(m):
    '''MSA representation [N_seq, L, c] -> single per-residue representation [L, c].'''
    # YOUR CODE HERE
    raise NotImplementedError

# --- checkpoint ---
m = torch.randn(Nseq, L, c)
s1 = pool_single(m)
assert s1.shape == (L, c), 'expected one vector per residue'
assert torch.allclose(s1, m.mean(0), atol=1e-6)
print('pool_single ok — the MSA collapses to one representation per residue ✓')

## Part 2 — assemble the whole model

The full forward pass, start to finish:

1. embed the MSA, run the **Evoformer trunk** (row/column attention + outer-product mean +
   triangle updates), seeded with the APC coevolution from rung 01;
2. read a **distogram** off the pair representation;
3. use that distogram as the structure module's pair conditioning (the trunk→structure
   bridge), and pool the MSA into the starting single representation;
4. from the **black-hole init**, run `NITER` iterations of IPA + frame update to fold a
   backbone.

The distogram bridge is deliberate: it forces the trunk to express its geometric knowledge
as *distances* (which we can also supervise directly), and lets the structure module fold
from the thing it is best at consuming. Gradients flow through the soft distogram, so the
two halves train together.

### Rep 2 — `fold(model, msa, apc)`
Wire the pipeline in `AlphaFold.fold`: given the pre-built submodules on `model`, run the
trunk, read the distogram, build the structure conditioning, and iterate the structure
module. Return `(R, t, traj, disto)`. Most lines are given; you fill the two handoffs
(pooling the single rep, and running the trunk).

In [ ]:
class AlphaFold(nn.Module):
    def __init__(self, nblk=2):
        super().__init__()
        self.emb = nn.Embedding(Q, c); self.pos = nn.Embedding(L, c)
        self.zpos = nn.Parameter(torch.zeros(L, L, cz)); self.zseed = nn.Linear(1, cz)
        self.row = nn.ModuleList([Row() for _ in range(nblk)]); self.col = nn.ModuleList([Col() for _ in range(nblk)])
        self.opm = nn.ModuleList([OPM() for _ in range(nblk)]); self.tri = nn.ModuleList([Tri() for _ in range(nblk)])
        self.disto = nn.Linear(cz, 16); self.zfold = nn.Linear(16, cz); self.to_single = nn.Linear(c, c)
        self.ipa = IPA(); self.ln = nn.LayerNorm(c); self.head = nn.Linear(c, 6)
    def trunk(self, msa, apc):
        m = self.emb(msa) + self.pos(torch.arange(L)); z = self.zpos + self.zseed(apc[..., None])
        for r, cl, o, t in zip(self.row, self.col, self.opm, self.tri):
            m = cl(r(m, z)); z = z + o(m); z = t(z); z = 0.5 * (z + z.transpose(0, 1))
        return m, z

def fold(model, msa, apc):
    '''Full MSA -> structure forward pass. Returns (R, t, trajectory, distogram_logits).'''
    # YOUR CODE HERE
    # hint:
    #   m, z = model.trunk(msa, apc)
    #   disto = model.disto(z)
    #   zf = model.zfold(F.softmax(disto, -1))                     # distogram -> structure conditioning
    #   x = model.to_single(pool_single(m)) + model.pos(torch.arange(L))
    #   R = torch.eye(3).expand(L,3,3).contiguous(); t = torch.zeros(L,3); traj = []
    #   for _ in range(NITER):
    #       x = model.ln(x + model.ipa(x, zf, R, t))
    #       u = model.head(x); R = R @ so3_exp(u[:, :3]*0.3); t = t + torch.einsum('lij,lj->li', R, u[:, 3:])
    #       traj.append((R, t))
    #   return R, t, traj, disto
    raise NotImplementedError

# --- checkpoint ---
net = AlphaFold()
msa, Xt, dbin, apc = train[0]
R, t, traj, disto = fold(net, msa, apc)
assert R.shape == (L, 3, 3) and t.shape == (L, 3), 'should output a frame + position per residue'
assert len(traj) == NITER and disto.shape == (L, L, 16), 'trajectory + distogram shapes'
assert torch.allclose(R @ R.transpose(-1, -2), torch.eye(3).expand(L, 3, 3), atol=1e-4), 'frames stay valid rotations'
print('fold ok — MSA in, a backbone (frames + coordinates) out ✓')

### Rep 3 — `total_loss(traj, disto, X_true, R_true, dbin_true)`
The end-to-end objective: **FAPE** averaged over the trajectory (deep supervision, rung 07)
plus the **distogram** cross-entropy (rung 04) as an auxiliary that keeps the trunk honest
about distances. Return `fape_mean + distogram_ce`.

In [ ]:
def total_loss(traj, disto, X_true, R_true, dbin_true):
    '''FAPE over the trajectory + distogram cross-entropy.'''
    # YOUR CODE HERE
    # hint: fape_mean = sum(fape(tt, RR, X_true, R_true) for RR, tt in traj) / len(traj)
    #       ce = F.cross_entropy(disto.reshape(-1, 16), dbin_true.reshape(-1))
    #       return fape_mean + ce
    raise NotImplementedError

# --- checkpoint ---
l = total_loss(traj, disto, Xt, Rtrain[0], dbin)
assert l.dim() == 0 and l.requires_grad and l.item() > 0
print('total loss ok — FAPE (structure) + distogram CE (trunk), one number to train on ✓')

## Part 4 — train the whole thing, and fold from an MSA

Train the complete pipeline end-to-end, then hand it held-out MSAs — proteins it has never
seen — and let it fold them from the alignment alone. We measure the aligned `Cα` RMSD
against a **random-fold baseline**, and look at the structures.

In [ ]:
def kabsch_rmsd(P, Qq):
    Pc = P - P.mean(0); Qc = Qq - Qq.mean(0); H = Pc.T @ Qc
    U, S, Vt = torch.linalg.svd(H); ds = torch.sign(torch.det(Vt.T @ U.T))
    Rr = Vt.T @ torch.diag(torch.tensor([1., 1., ds])) @ U.T
    return ((Rr @ Pc.T).T - Qc).norm(dim=-1).pow(2).mean().sqrt()

opt = torch.optim.AdamW(net.parameters(), lr=2e-3)
t0 = time.time(); hist = []
for step in range(2600):
    idx = int(rng.integers(len(train)))
    msa, Xt, dbin, apc = train[idx]
    R, t, traj, disto = fold(net, msa, apc)
    loss = total_loss(traj, disto, Xt, Rtrain[idx], dbin)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 200 == 0: hist.append((step, loss.item()))
print('trained the whole pipeline in %.0fs' % (time.time() - t0))

### Rep 4 — `fold_vs_random(rmsds, random_rmsds)`
Return the mean predicted RMSD as a fraction of the mean random-fold RMSD. Below `1.0` means
the pipeline genuinely folds; we want it clearly below.

In [ ]:
def fold_vs_random(rmsds, random_rmsds):
    '''mean(predicted RMSD) / mean(random-fold RMSD).'''
    # YOUR CODE HERE
    raise NotImplementedError

# --- checkpoint ---
net.eval()
with torch.no_grad():
    rmsds = [kabsch_rmsd(fold(net, te[0], te[3])[1], te[1]).item() for te in test]
    random_rmsds = [kabsch_rmsd(torch.tensor(make_structure()), te[1]).item() for te in test]
ratio = fold_vs_random(rmsds, random_rmsds)
assert ratio < 0.85, 'the end-to-end model must fold clearly better than random'
print('END-TO-END (MSA -> structure), held out:')
print('  aligned Ca-RMSD  %.2f   vs random-fold baseline  %.2f   ->  %.0f%% of random'
      % (np.mean(rmsds), np.mean(random_rmsds), 100 * ratio))
with torch.no_grad():
    dacc = np.mean([(fold(net, te[0], te[3])[3].argmax(-1) == te[2]).float().mean().item() for te in test])
print('  (trunk distogram top-1 accuracy %.2f — Track A feeding Track B)' % dacc)
print('\nFed only an alignment, the full pipeline folds a recognisable backbone. ✓')

fig, ax = plt.subplots(figsize=(5.2, 3.2))
hh = np.array(hist); ax.plot(hh[:, 0], hh[:, 1], color=BLUE, lw=2)
ax.set_xlabel('step'); ax.set_ylabel('FAPE + distogram loss'); ax.set_title('training a toy AlphaFold end-to-end'); ax.grid(alpha=.15)
plt.show()

In [ ]:
# The money shot: fold three held-out proteins from their MSA, predicted (blue) vs true (green).
def superpose(P, Qq):
    Pc = P - P.mean(0); Qc = Qq - Qq.mean(0); H = Pc.T @ Qc
    U, S, Vt = torch.linalg.svd(H); ds = torch.sign(torch.det(Vt.T @ U.T))
    Rr = Vt.T @ torch.diag(torch.tensor([1., 1., ds])) @ U.T
    return (Rr @ Pc.T).T, Qc

fig = plt.figure(figsize=(12, 3.8))
with torch.no_grad():
    for k in range(3):
        te = test[k]; _, t, _, _ = fold(net, te[0], te[3]); P, Qc = superpose(t, te[1])
        ax = fig.add_subplot(1, 3, k + 1, projection='3d')
        ax.plot(*Qc.T.numpy(), '-o', ms=3, color=GREEN, lw=1.5, label='true')
        ax.plot(*P.T.numpy(), '-o', ms=3, color=BLUE, lw=1.5, label='folded from MSA')
        ax.set_title('held-out protein %d (RMSD %.2f)' % (k, kabsch_rmsd(t, te[1])), fontsize=9)
        ax.set_xticklabels([]); ax.set_yticklabels([]); ax.set_zticklabels([])
        if k == 0: ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
print('An MSA went in; these backbones came out. That is AlphaFold, in miniature. ✓')

## Reflection — what just transferred

- **You built AlphaFold.** Not a wrapper around someone's weights — the actual machinery,
  from the coevolution insight (01) through axial attention (02), triangle operations (03),
  the distogram (04), residue frames (05), invariant point attention (06), frame updates and
  FAPE (07), to recycling and confidence (08). This notebook wired them into one model that
  folds an MSA into 3D.
- **The distogram is the bridge** between the two tracks: the trunk expresses geometry as
  distances; the structure module realises those distances as coordinates; gradients cross
  the bridge so both halves learn together.
- **It genuinely folds from sequence.** Given only an alignment of a protein it never saw,
  the pipeline produces a backbone well below the random-fold RMSD. It is not sub-ångström —
  that is a story of scale (huge models, deep MSAs, months of TPU training, the full
  atom-level structure module, and templates). But the ideas are all here, and they are the
  same ideas.
- **What the real system adds on top of what you built:** far more Evoformer blocks and
  channels; the full all-atom structure module with sidechain torsions; templates; a much
  larger MSA featurization; the confidence heads as trained distributions; and a training
  regime with self-distillation. Each is an extension of a mechanism in this series — none is
  a new idea you have not met.

**Next rung:** `AF2·10 — Dissect real AlphaFold2`. Run ColabFold on a real protein sequence
and read its outputs — pLDDT, PAE, the recycles — mapping every plot back to the mechanism in
this series that produced it. The one notebook that needs a GPU, and the one that connects
everything you built to the model people actually use.

---
Scroll down only after you've done the reps.

## Solutions appendix (peek only after trying)

In [ ]:
def pool_single(m):
    return m.mean(0)

def fold(model, msa, apc):
    m, z = model.trunk(msa, apc)
    disto = model.disto(z)
    zf = model.zfold(F.softmax(disto, -1))                       # distogram bridge, differentiable
    x = model.to_single(pool_single(m)) + model.pos(torch.arange(L))
    R = torch.eye(3).expand(L, 3, 3).contiguous(); t = torch.zeros(L, 3); traj = []
    for _ in range(NITER):
        x = model.ln(x + model.ipa(x, zf, R, t))
        u = model.head(x); R = R @ so3_exp(u[:, :3] * 0.3); t = t + torch.einsum('lij,lj->li', R, u[:, 3:])
        traj.append((R, t))
    return R, t, traj, disto

def total_loss(traj, disto, X_true, R_true, dbin_true):
    fape_mean = sum(fape(tt, RR, X_true, R_true) for RR, tt in traj) / len(traj)
    ce = F.cross_entropy(disto.reshape(-1, 16), dbin_true.reshape(-1))
    return fape_mean + ce

def fold_vs_random(rmsds, random_rmsds):
    return float(np.mean(rmsds)) / float(np.mean(random_rmsds))

print('reference solutions loaded — re-run the checkpoint cells above')